# Graph-To-Vec Classification Walkthrough

This notebook walks through the v1 framework from heterogeneous NetworkX data to graph-level classification, node-level classification, metapath embeddings, and CLI-ready artifacts.

## 1. Local Setup

Run this notebook with the project virtual environment. The project intentionally uses a local `.venv`, not a container. From the repo root:

```bash
uv venv --python /opt/homebrew/bin/python3.11 .venv
uv pip install -r requirements.txt
uv pip install -e .
uv pip install -r requirements-notebook.txt
```

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "src" / "graph_to_vec").exists():
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT))
print(ROOT)

In [ ]:
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split

from examples.end_to_end_classification import (
    graphs_to_tables,
    make_transaction_dataset,
    make_user_network,
    write_cli_fixture,
)
from graph_to_vec import (
    Graph2VecTransformer,
    GraphClassificationPipeline,
    GraphSchema,
    MetaPath2VecNodeEmbedder,
    NodeClassifierTrainer,
    from_networkx,
    from_tables,
    to_networkx,
)
from graph_to_vec.persistence import load_joblib, save_joblib

RANDOM_STATE = 11
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)
RUN_DIR = ROOT / "runs" / "notebook_walkthrough"
RUN_DIR.mkdir(parents=True, exist_ok=True)

## 2. Make A Tiny Heterogeneous Graph Dataset

Each sample is a transaction-shaped `networkx.MultiDiGraph` with typed nodes, typed relations, numeric node features, and a graph label.

In [ ]:
graphs, labels = make_transaction_dataset(count=60)
sample = graphs[1]

print(f"graphs: {len(graphs)}")
print(f"labels: {pd.Series(labels).value_counts().to_dict()}")
print(f"sample graph id: {sample.graph['id']}")
print(f"sample graph label: {sample.graph['label']}")
print(f"sample nodes: {list(sample.nodes(data=True))[:2]}")
print(f"sample edges: {list(sample.edges(data=True))[:3]}")

## 3. Convert NetworkX To PyG HeteroData

`from_networkx` preserves original ids and relation structure while producing the internal training representation used by PyG models.

In [ ]:
data = from_networkx(sample)

print("node types:", data.node_types)
print("edge types:", data.edge_types)
print("customer ids:", data["customer"].original_id)
print("customer feature shape:", tuple(data["customer"].x.shape))

roundtrip = to_networkx(data)
print("round-tripped nodes:", roundtrip.number_of_nodes())
print("round-tripped edges:", roundtrip.number_of_edges())

## 4. Graph-Level Classification

`GraphClassificationPipeline` combines typed WL/Graph2Vec embeddings with a normal sklearn classifier. The embedder is also usable directly in sklearn pipelines and grid search.

In [ ]:
train_graphs, test_graphs, y_train, y_test = train_test_split(
    graphs,
    labels,
    test_size=0.30,
    random_state=RANDOM_STATE,
    stratify=labels,
)

pipeline = GraphClassificationPipeline(
    embedder=Graph2VecTransformer(
        iterations=2,
        embedding_dim=64,
        include_features=True,
        random_state=RANDOM_STATE,
    )
)
pipeline.fit(train_graphs, y_train)
predictions = pipeline.predict(test_graphs)

report = pd.DataFrame(
    classification_report(y_test, predictions, output_dict=True, zero_division=0)
).T
report.round(3)

In [ ]:
preview = pd.DataFrame(
    {
        "graph_id": [graph.graph["id"] for graph in test_graphs],
        "true": y_test,
        "predicted": predictions,
    }
)
preview.head(10)

## 5. Persist And Reload The Pipeline

The persistence helpers write normal sklearn/joblib artifacts for graph embedding pipelines.

In [ ]:
artifact_path = RUN_DIR / "graph_pipeline.joblib"
save_joblib(pipeline, artifact_path)
reloaded = load_joblib(artifact_path)

assert reloaded.predict(test_graphs).tolist() == predictions.tolist()
artifact_path

## 6. Table Adapter

For repeatable CLI workflows, graph samples can come from node and edge tables plus a `GraphSchema`.

In [ ]:
nodes, edges = graphs_to_tables(graphs)
schema = GraphSchema(graph_id_col="graph_id", graph_label_col="graph_label")
table_graphs = from_tables(nodes, edges, schema)

print(f"node rows: {len(nodes)}")
print(f"edge rows: {len(edges)}")
print(f"converted graph samples: {len(table_graphs)}")
print(f"first graph label: {table_graphs[0].graph_label}")
nodes.head()

## 7. Node-Level Classification

For node classification, labels and masks live on the target node type. This example trains a small CPU HeteroSAGE model over one heterogeneous user network.

In [ ]:
user_graph = make_user_network(count=36)
user_data = from_networkx(user_graph)

trainer = NodeClassifierTrainer(
    target_node_type="user",
    epochs=40,
    lr=0.04,
    device="cpu",
)
trainer.fit(user_data)

test_mask = user_data["user"].test_mask
metrics = trainer.evaluate(user_data, mask=test_mask)
predicted = trainer.predict(user_data, mask=test_mask)
truth = user_data["user"].y[test_mask].detach().cpu().numpy()
user_ids = np.asarray(user_data["user"].original_id, dtype=object)[test_mask.detach().cpu().numpy()]

metrics, pd.DataFrame({"user_id": user_ids, "true": truth, "predicted": predicted})

## 8. Metapath Node Embeddings

`MetaPath2VecNodeEmbedder` creates node vectors from typed random walks. The configured metapath below alternates from user to device and back to user.

In [ ]:
metapath_embedder = MetaPath2VecNodeEmbedder(
    metapaths=[[('user', 'uses_device', 'device'), ('device', 'used_by', 'user')]],
    walk_length=8,
    walks_per_node=4,
    embedding_dim=8,
    random_state=RANDOM_STATE,
)
node_embeddings = metapath_embedder.fit_transform(user_data)
{node_type: matrix.shape for node_type, matrix in node_embeddings.items()}

## 9. Generate CLI Fixtures

The example helper writes CSVs and a YAML config under `runs/example_walkthrough/`. Use that config with the `g2v` CLI for repeatable train/evaluate/predict runs.

In [ ]:
config_path = write_cli_fixture(graphs)
print(f"train:    .venv/bin/g2v train --config {config_path}")
print(f"evaluate: .venv/bin/g2v evaluate --config {config_path}")
print(f"predict:  .venv/bin/g2v predict --config {config_path}")